# Attendee Crossover Analysis

This notebook compares attendees between two data sources:
- **Wix** (current events from Dec 2025+)
- **Eventbrite** (historical events from Nov 2022+)

We'll use both exact email matching and probabilistic name matching to identify audience crossover.


## 1. Data Loading and Exploration


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import glob
import re

# For fuzzy matching
from rapidfuzz import fuzz, process

# For visualizations
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')


In [ ]:
# Find the latest Wix ticket sales file
DATA_DIR = Path('../data')
WIX_DIR = DATA_DIR / 'views'
EVENTBRITE_FILE = DATA_DIR / 'other' / 'invite_only_cleaned.csv'

def find_latest_csv(directory: Path, prefix: str) -> Path | None:
    """Find the most recently modified CSV file with the given prefix."""
    pattern = str(directory / f"{prefix}_*.csv")
    files = glob.glob(pattern)
    if not files:
        return None
    return Path(sorted(files, key=lambda x: Path(x).stat().st_mtime, reverse=True)[0])

wix_file = find_latest_csv(WIX_DIR, 'ticket_sales')
print(f"Wix file: {wix_file}")
print(f"Eventbrite file: {EVENTBRITE_FILE}")


In [ ]:
# Load both datasets
wix_raw = pd.read_csv(wix_file)
eventbrite_raw = pd.read_csv(EVENTBRITE_FILE)

print(f"Wix data: {len(wix_raw):,} rows, {len(wix_raw.columns)} columns")
print(f"Eventbrite data: {len(eventbrite_raw):,} rows, {len(eventbrite_raw.columns)} columns")


In [ ]:
# Preview Wix data
print("=" * 60)
print("WIX DATA COLUMNS")
print("=" * 60)
print(wix_raw.columns.tolist())
wix_raw.head(3)


In [ ]:
# Preview Eventbrite data
print("=" * 60)
print("EVENTBRITE DATA COLUMNS")
print("=" * 60)
print(eventbrite_raw.columns.tolist())
eventbrite_raw.head(3)


In [ ]:
def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize column names to lowercase snake_case."""
    df = df.copy()
    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(' ', '_', regex=False)
        .str.replace('-', '_', regex=False)
        .str.replace('/', '_', regex=False)
    )
    return df

wix = standardize_columns(wix_raw)
eventbrite = standardize_columns(eventbrite_raw)

print("Wix columns after standardization:")
print(wix.columns.tolist())
print("\nEventbrite columns after standardization:")
print(eventbrite.columns.tolist())


In [ ]:
# Create normalized versions of key fields
def normalize_email(email):
    """Normalize email: lowercase, strip whitespace."""
    if pd.isna(email):
        return None
    return str(email).lower().strip()

def normalize_name(name):
    """Normalize name: lowercase, strip, remove extra spaces."""
    if pd.isna(name):
        return ''
    return re.sub(r'\s+', ' ', str(name).lower().strip())

# Apply to Wix data
wix['email_normalized'] = wix['buyer_email'].apply(normalize_email)
wix['first_name_normalized'] = wix['buyer_first_name'].apply(normalize_name)
wix['last_name_normalized'] = wix['buyer_last_name'].apply(normalize_name)
wix['full_name_normalized'] = wix['first_name_normalized'] + ' ' + wix['last_name_normalized']
wix['full_name_normalized'] = wix['full_name_normalized'].str.strip()

# Apply to Eventbrite data
eventbrite['email_normalized'] = eventbrite['buyer_email'].apply(normalize_email)
eventbrite['first_name_normalized'] = eventbrite['buyer_first_name'].apply(normalize_name)
eventbrite['last_name_normalized'] = eventbrite['buyer_last_name'].apply(normalize_name)
eventbrite['full_name_normalized'] = eventbrite['first_name_normalized'] + ' ' + eventbrite['last_name_normalized']
eventbrite['full_name_normalized'] = eventbrite['full_name_normalized'].str.strip()

print("Sample normalized data:")
wix[['buyer_email', 'email_normalized', 'full_name_normalized']].head()


In [ ]:
# Get unique attendees from each platform
wix_attendees = wix.groupby('email_normalized').agg({
    'full_name_normalized': 'first',
    'buyer_first_name': 'first',
    'buyer_last_name': 'first',
    'event_name': 'nunique',
    'order_number': 'nunique'
}).reset_index()
wix_attendees.columns = ['email', 'full_name', 'first_name', 'last_name', 'events_attended', 'orders']
wix_attendees['source'] = 'wix'

eventbrite_attendees = eventbrite.groupby('email_normalized').agg({
    'full_name_normalized': 'first',
    'buyer_first_name': 'first',
    'buyer_last_name': 'first',
    'event_name': 'nunique',
    'order_id': 'nunique'
}).reset_index()
eventbrite_attendees.columns = ['email', 'full_name', 'first_name', 'last_name', 'events_attended', 'orders']
eventbrite_attendees['source'] = 'eventbrite'

print(f"Unique Wix attendees (by email): {len(wix_attendees):,}")
print(f"Unique Eventbrite attendees (by email): {len(eventbrite_attendees):,}")


In [ ]:
# Date range analysis
print("=" * 60)
print("DATE RANGES")
print("=" * 60)

# Wix dates
wix['order_date_parsed'] = pd.to_datetime(wix['order_date'], errors='coerce')
wix['event_start_parsed'] = pd.to_datetime(wix['event_start_date'], errors='coerce')

print(f"\nWix Order Dates: {wix['order_date_parsed'].min()} to {wix['order_date_parsed'].max()}")
print(f"Wix Event Dates: {wix['event_start_parsed'].min()} to {wix['event_start_parsed'].max()}")

# Eventbrite dates
eventbrite['order_date_parsed'] = pd.to_datetime(eventbrite['order_date'], errors='coerce')
eventbrite['event_start_parsed'] = pd.to_datetime(eventbrite['event_start_date'], errors='coerce')

print(f"\nEventbrite Order Dates: {eventbrite['order_date_parsed'].min()} to {eventbrite['order_date_parsed'].max()}")
print(f"Eventbrite Event Dates: {eventbrite['event_start_parsed'].min()} to {eventbrite['event_start_parsed'].max()}")


In [ ]:
# Unique events in each platform
print("=" * 60)
print("UNIQUE EVENTS")
print("=" * 60)

wix_events = wix['event_name'].unique()
eventbrite_events = eventbrite['event_name'].unique()

print(f"\nWix Events ({len(wix_events)}):")
for e in sorted(wix_events):
    count = len(wix[wix['event_name'] == e])
    print(f"  - {e} ({count} tickets)")

print(f"\nEventbrite Events ({len(eventbrite_events)}):")
for e in sorted(eventbrite_events):
    count = len(eventbrite[eventbrite['event_name'] == e])
    print(f"  - {e} ({count} orders)")


## 2. Simple Matching (Exact Email)


In [ ]:
# Find exact email matches
wix_emails = set(wix_attendees['email'].dropna())
eventbrite_emails = set(eventbrite_attendees['email'].dropna())

# Calculate overlaps
exact_matches = wix_emails & eventbrite_emails
wix_only = wix_emails - eventbrite_emails
eventbrite_only = eventbrite_emails - wix_emails

print("=" * 60)
print("EXACT EMAIL MATCHING RESULTS")
print("=" * 60)
print(f"\nTotal unique Wix emails: {len(wix_emails):,}")
print(f"Total unique Eventbrite emails: {len(eventbrite_emails):,}")
print(f"\nExact email matches: {len(exact_matches):,}")
print(f"Wix-only attendees: {len(wix_only):,}")
print(f"Eventbrite-only attendees: {len(eventbrite_only):,}")

# Calculate percentages
pct_wix_from_eventbrite = (len(exact_matches) / len(wix_emails)) * 100 if wix_emails else 0
pct_eventbrite_returned = (len(exact_matches) / len(eventbrite_emails)) * 100 if eventbrite_emails else 0

print(f"\n{pct_wix_from_eventbrite:.1f}% of Wix attendees were previously on Eventbrite")
print(f"{pct_eventbrite_returned:.1f}% of Eventbrite attendees have returned via Wix")


In [ ]:
# Create a dataframe of matched attendees
matched_attendees = wix_attendees[wix_attendees['email'].isin(exact_matches)].copy()
matched_attendees = matched_attendees.merge(
    eventbrite_attendees[['email', 'events_attended', 'orders']],
    on='email',
    suffixes=('_wix', '_eventbrite')
)
matched_attendees = matched_attendees.rename(columns={
    'events_attended': 'wix_events',
    'orders': 'wix_orders',
    'events_attended_eventbrite': 'eventbrite_events',
    'orders_eventbrite': 'eventbrite_orders'
})

print(f"\nMatched attendees with event counts:")
matched_attendees[['email', 'full_name', 'wix_events', 'eventbrite_events']].head(10)


In [ ]:
# Show the full list of crossover attendees
print(f"\n" + "=" * 60)
print(f"ALL {len(matched_attendees)} CROSSOVER ATTENDEES (Email Match)")
print("=" * 60)
matched_attendees.sort_values('eventbrite_events', ascending=False)


## 3. Probabilistic Matching (Names)

For attendees who may have used different email addresses, we'll use fuzzy name matching.


In [ ]:
# Get attendees without email matches for fuzzy name matching
wix_no_match = wix_attendees[~wix_attendees['email'].isin(exact_matches)].copy()
eventbrite_no_match = eventbrite_attendees[~eventbrite_attendees['email'].isin(exact_matches)].copy()

print(f"Wix attendees without email match: {len(wix_no_match):,}")
print(f"Eventbrite attendees without email match: {len(eventbrite_no_match):,}")


In [ ]:
def fuzzy_match_names(
    source_df: pd.DataFrame, 
    target_df: pd.DataFrame,
    threshold: int = 85,
    name_col: str = 'full_name'
) -> pd.DataFrame:
    """
    Find fuzzy name matches between source and target dataframes.
    
    Args:
        source_df: DataFrame to match FROM (e.g., Wix)
        target_df: DataFrame to match TO (e.g., Eventbrite)
        threshold: Minimum fuzzy match score (0-100)
        name_col: Column containing names to match
    
    Returns:
        DataFrame with matched pairs and scores
    """
    matches = []
    
    # Get unique target names for matching
    target_names = target_df[name_col].dropna().unique().tolist()
    
    # Skip if empty
    if not target_names:
        return pd.DataFrame()
    
    for _, row in source_df.iterrows():
        source_name = row[name_col]
        source_email = row['email']
        
        # Skip empty names
        if pd.isna(source_name) or not source_name.strip():
            continue
        
        # Find best match
        result = process.extractOne(
            source_name, 
            target_names,
            scorer=fuzz.token_sort_ratio
        )
        
        if result and result[1] >= threshold:
            matched_name, score, _ = result
            
            # Get the matched row from target
            target_row = target_df[target_df[name_col] == matched_name].iloc[0]
            
            matches.append({
                'source_name': source_name,
                'source_email': source_email,
                'target_name': matched_name,
                'target_email': target_row['email'],
                'match_score': score,
                'match_type': 'exact' if score == 100 else 'fuzzy'
            })
    
    return pd.DataFrame(matches)


In [ ]:
# Run fuzzy matching (may take a moment with large datasets)
print("Running fuzzy name matching...")
print(f"Matching {len(wix_no_match)} Wix names against {len(eventbrite_no_match)} Eventbrite names...")

fuzzy_matches = fuzzy_match_names(
    wix_no_match, 
    eventbrite_no_match,
    threshold=85
)

print(f"\nFound {len(fuzzy_matches)} potential name matches")


In [ ]:
# Display fuzzy matches sorted by score
if len(fuzzy_matches) > 0:
    print("=" * 60)
    print("FUZZY NAME MATCHES (score >= 85)")
    print("=" * 60)
    display(fuzzy_matches.sort_values('match_score', ascending=False))
else:
    print("No additional fuzzy matches found beyond email matches.")


In [ ]:
# Filter for high-confidence matches (score >= 90)
high_confidence_matches = fuzzy_matches[fuzzy_matches['match_score'] >= 90] if len(fuzzy_matches) > 0 else pd.DataFrame()

print(f"High-confidence name matches (score >= 90): {len(high_confidence_matches)}")

if len(high_confidence_matches) > 0:
    print("\nThese are likely the same person with different emails:")
    display(high_confidence_matches)


In [ ]:
# Combined matching summary
print("=" * 60)
print("COMBINED MATCHING SUMMARY")
print("=" * 60)

total_email_matches = len(exact_matches)
total_fuzzy_matches = len(high_confidence_matches) if len(high_confidence_matches) > 0 else 0
total_crossover = total_email_matches + total_fuzzy_matches

print(f"\nExact email matches: {total_email_matches}")
print(f"High-confidence name matches: {total_fuzzy_matches}")
print(f"\nTotal identified crossover: {total_crossover} attendees")
print(f"  - From Wix attendee pool: {len(wix_emails):,}")
print(f"  - Match rate: {(total_crossover / len(wix_emails)) * 100:.1f}%")


## 4. Crossover Analysis


In [ ]:
# Analyze which Eventbrite events the crossover attendees came from
crossover_emails = list(exact_matches)

# Get their Eventbrite history
eventbrite_history = eventbrite[eventbrite['email_normalized'].isin(crossover_emails)]

print("=" * 60)
print("EVENTBRITE HISTORY OF CROSSOVER ATTENDEES")
print("=" * 60)

event_counts = eventbrite_history['event_name'].value_counts()
print(f"\nEvents these returning attendees previously attended:")
print(event_counts.to_string())


In [ ]:
# Analyze which Wix events crossover attendees are now attending
wix_current = wix[wix['email_normalized'].isin(crossover_emails)]

print("\n" + "=" * 60)
print("WIX EVENTS CROSSOVER ATTENDEES ARE NOW ATTENDING")
print("=" * 60)

wix_event_counts = wix_current['event_name'].value_counts()
print(f"\nEvents returning attendees are attending on Wix:")
print(wix_event_counts.to_string())


In [ ]:
# Frequency analysis: Most frequent crossover attendees
print("\n" + "=" * 60)
print("MOST ENGAGED CROSSOVER ATTENDEES")
print("=" * 60)

crossover_engagement = matched_attendees.copy()
crossover_engagement['total_events'] = crossover_engagement['wix_events'] + crossover_engagement['eventbrite_events']
crossover_engagement = crossover_engagement.sort_values('total_events', ascending=False)

print("\nTop 15 most engaged crossover attendees:")
crossover_engagement[['full_name', 'email', 'wix_events', 'eventbrite_events', 'total_events']].head(15)


In [ ]:
# Event-to-event crossover matrix
# Which Eventbrite events lead to which Wix events?

print("\n" + "=" * 60)
print("EVENT-TO-EVENT CROSSOVER ANALYSIS")
print("=" * 60)

# For each crossover attendee, create pairs of (eventbrite_event, wix_event)
event_pairs = []

for email in crossover_emails:
    eb_events = eventbrite_history[eventbrite_history['email_normalized'] == email]['event_name'].unique()
    wix_events_for_email = wix_current[wix_current['email_normalized'] == email]['event_name'].unique()
    
    for eb_event in eb_events:
        for wix_event in wix_events_for_email:
            event_pairs.append({
                'eventbrite_event': eb_event,
                'wix_event': wix_event,
                'email': email
            })

event_pairs_df = pd.DataFrame(event_pairs)

if len(event_pairs_df) > 0:
    # Count pairs
    pair_counts = event_pairs_df.groupby(['eventbrite_event', 'wix_event']).size().reset_index(name='count')
    pair_counts = pair_counts.sort_values('count', ascending=False)
    
    print("\nMost common event-to-event transitions:")
    display(pair_counts.head(20))
else:
    print("No event pairs found for crossover analysis.")


## 5. Visualizations


In [ ]:
# Venn diagram of attendee overlap
fig, ax = plt.subplots(figsize=(10, 8))

venn = venn2(
    [wix_emails, eventbrite_emails],
    set_labels=('Wix', 'Eventbrite'),
    ax=ax
)

# Style the venn diagram
for text in venn.set_labels:
    if text:
        text.set_fontsize(14)
        text.set_fontweight('bold')

for text in venn.subset_labels:
    if text:
        text.set_fontsize(12)

plt.title('Attendee Overlap: Wix vs Eventbrite', fontsize=16, fontweight='bold', pad=20)

# Add summary text
summary_text = f"""Overlap: {len(exact_matches)} attendees
{pct_wix_from_eventbrite:.1f}% of Wix from Eventbrite
{pct_eventbrite_returned:.1f}% of Eventbrite returned"""
plt.text(0.5, -0.1, summary_text, transform=ax.transAxes, fontsize=11, 
         ha='center', va='top', style='italic')

plt.tight_layout()
plt.show()


In [ ]:
# Bar chart: Eventbrite events that produced crossover attendees
if len(event_counts) > 0:
    fig, ax = plt.subplots(figsize=(12, max(6, len(event_counts) * 0.4)))
    
    event_counts_sorted = event_counts.sort_values(ascending=True)
    
    bars = ax.barh(range(len(event_counts_sorted)), event_counts_sorted.values)
    ax.set_yticks(range(len(event_counts_sorted)))
    ax.set_yticklabels(event_counts_sorted.index, fontsize=10)
    ax.set_xlabel('Number of Returning Attendees', fontsize=12)
    ax.set_title('Eventbrite Events That Produced Returning Wix Attendees', 
                 fontsize=14, fontweight='bold', pad=15)
    
    # Add value labels on bars
    for i, (bar, val) in enumerate(zip(bars, event_counts_sorted.values)):
        ax.text(val + 0.1, i, str(val), va='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()
else:
    print("No crossover attendees to visualize.")


In [ ]:
# Heatmap of event crossover (if we have pairs)
if len(event_pairs_df) > 0 and len(pair_counts) >= 4:
    # Create a pivot table for the heatmap
    pivot = pair_counts.pivot_table(
        index='eventbrite_event', 
        columns='wix_event', 
        values='count',
        fill_value=0
    )
    
    fig, ax = plt.subplots(figsize=(14, max(8, len(pivot) * 0.5)))
    
    sns.heatmap(
        pivot, 
        annot=True, 
        fmt='d', 
        cmap='YlOrRd',
        ax=ax,
        cbar_kws={'label': 'Number of Attendees'}
    )
    
    ax.set_xlabel('Wix Events (Current)', fontsize=12)
    ax.set_ylabel('Eventbrite Events (Historical)', fontsize=12)
    ax.set_title('Event Crossover Heatmap\n(Attendees from Eventbrite -> Wix)', 
                 fontsize=14, fontweight='bold', pad=15)
    
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough event pair data to create a meaningful heatmap.")


In [ ]:
# Summary statistics plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Attendee breakdown
categories = ['Wix Only', 'Eventbrite Only', 'Both Platforms']
counts = [len(wix_only), len(eventbrite_only), len(exact_matches)]
colors = ['#4ECDC4', '#FF6B6B', '#45B7D1']

axes[0].bar(categories, counts, color=colors)
axes[0].set_ylabel('Number of Attendees')
axes[0].set_title('Attendee Platform Distribution', fontweight='bold')
for i, v in enumerate(counts):
    axes[0].text(i, v + max(counts)*0.02, str(v), ha='center', fontweight='bold')

# Plot 2: Crossover rate as pie chart
sizes = [len(exact_matches), len(wix_emails) - len(exact_matches)]
labels = [f'Returning\n({len(exact_matches)})', f'New\n({len(wix_emails) - len(exact_matches)})']
colors_pie = ['#45B7D1', '#95E1D3']

axes[1].pie(sizes, labels=labels, colors=colors_pie, autopct='%1.1f%%', 
            startangle=90, explode=(0.05, 0))
axes[1].set_title('Wix Attendees: Returning vs New', fontweight='bold')

# Plot 3: Engagement comparison
if len(matched_attendees) > 0:
    engagement_data = {
        'Platform': ['Wix', 'Eventbrite'],
        'Avg Events': [
            matched_attendees['wix_events'].mean(),
            matched_attendees['eventbrite_events'].mean()
        ]
    }
    axes[2].bar(engagement_data['Platform'], engagement_data['Avg Events'], 
                color=['#4ECDC4', '#FF6B6B'])
    axes[2].set_ylabel('Average Events per Attendee')
    axes[2].set_title('Crossover Attendee Engagement', fontweight='bold')
    for i, v in enumerate(engagement_data['Avg Events']):
        axes[2].text(i, v + 0.05, f'{v:.2f}', ha='center', fontweight='bold')
else:
    axes[2].text(0.5, 0.5, 'No data', ha='center', va='center')
    axes[2].set_title('Engagement (No crossover data)', fontweight='bold')

plt.tight_layout()
plt.show()


## Summary

This analysis identified:
1. **Exact email matches** between Wix and Eventbrite attendees
2. **Probabilistic name matches** for attendees who may have used different emails
3. **Event crossover patterns** showing which historical events lead to current engagement

Key findings are displayed in the visualizations above.


In [ ]:
# Final summary table
print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

summary_data = {
    'Metric': [
        'Total Wix Attendees (unique email)',
        'Total Eventbrite Attendees (unique email)',
        'Exact Email Matches',
        'High-Confidence Name Matches (score >= 90)',
        'Total Identified Crossover',
        '% of Wix from Eventbrite',
        '% of Eventbrite returned to Wix'
    ],
    'Value': [
        f"{len(wix_emails):,}",
        f"{len(eventbrite_emails):,}",
        f"{len(exact_matches):,}",
        f"{len(high_confidence_matches) if len(high_confidence_matches) > 0 else 0}",
        f"{total_crossover:,}",
        f"{pct_wix_from_eventbrite:.1f}%",
        f"{pct_eventbrite_returned:.1f}%"
    ]
}

summary_df = pd.DataFrame(summary_data)
display(summary_df)
